In [1]:
from zenml.pipelines import pipeline
from zenml.steps import step

In [2]:
@pipeline(enable_cache=False)
def classification_pipeline():
    """Animal classification pipeline with data loading, training, evaluation, and deployment."""
    # Import the mlflow deployer step
    from zenml.integrations.mlflow.steps import mlflow_model_deployer_step

    # Load data
    # train_dataloader, val_dataloader, test_dataloader = load_data()
    train_dataloader, validation_dataloader, test_dataloader = load_data()

    # Train model
    trained_model = model_training(
        train_dataloader=train_dataloader,
        val_dataloader=validation_dataloader,
    )

    # Evaluate model
    accuracy, precision, recall, f1 = model_evaluation(
        model=trained_model,
        test_loader=test_dataloader,
    )

    # Check deployment criteria
    deployment_decision = deployment_trigger(
        accuracy=accuracy,
        precision=precision,
        recall=recall,
        f1=f1,
    )

    # Deploy model if criteria met - use named parameters
    # Note: model_name must match the artifact_path used in mlflow.pytorch.log_model
    mlflow_model_deployer_step(
        deploy_decision=deployment_decision,
        model=trained_model,
        model_name="animal-classifier-resnet18",
        workers=2,
    )


In [3]:
from typing import Tuple

import yaml
from torch.utils.data import DataLoader
from zenml.integrations.pytorch.materializers import PyTorchDataLoaderMaterializer

# from zenml.materializers import PydanticMaterializer
from src.config import DataConfig
from src.data.dataset import (
    build_data_loaders,
    prepare_data_for_training,
)


@step(output_materializers=PyTorchDataLoaderMaterializer)
# @step(output_materializers=PydanticMaterializer)
def load_data() -> Tuple[DataLoader, DataLoader, DataLoader]:
    with open("src/steps/test_config.yaml", "r") as f:
        config = yaml.safe_load(f)
        data_config = DataConfig(**config["data"])

    dataset_bundle = prepare_data_for_training(data_config)
    dataloader = build_data_loaders(
        dataset_bundle,
        batch_size=data_config.batch_size,
        num_workers=data_config.num_workers,
    )

    num_classes = len(dataset_bundle.label_names)
    # add num_classes to the config
    config["data"]["num_classes"] = num_classes
    with open("src/steps/test_config.yaml", "w") as f:
        yaml.dump(config, f)

    # print(config)

    # return dataloaders["train"], dataloaders["validation"], dataloaders["test"]
    return (
        dataloader["train_dataloader"],
        dataloader["validation_dataloader"],
        dataloader["test_dataloader"],
    )


In [4]:
import mlflow
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import DataLoader
from mlflow.models import infer_signature
from zenml.integrations.pytorch.materializers import PyTorchModuleMaterializer

from src.config import TrainConfig
from src.models.resnet18 import AnimalClassifierResNet18


@step(
    output_materializers=PyTorchModuleMaterializer,
    experiment_tracker="mlflow_tracker",
    enable_cache=False,
)
def model_training(
    train_dataloader: DataLoader,
    val_dataloader: DataLoader,
) -> nn.Module:
    with open("src/steps/test_config.yaml", "r") as f:
        config = yaml.safe_load(f)
        data_config = DataConfig(**config["data"])
        print(
            data_config.num_classes
        ) if data_config.num_classes else "No num_classes found in config"
        train_config = TrainConfig(**config["train"])

    # Log training parameters to MLflow
    mlflow.log_params(train_config.model_dump(mode="json"))

    model = AnimalClassifierResNet18(
        num_classes=data_config.num_classes,
        optimizer=train_config.optimizer,
        pretrained=train_config.pretrained,
        lr=train_config.lr,
        max_lr=train_config.max_lr,
        epochs=train_config.epochs,
        device=train_config.device,
        train_loader=train_dataloader,
        class_weights=None,
    )

    model.fit(
        train_loader=train_dataloader,
        val_loader=val_dataloader,
        save_dir=train_config.save_dir,
        save_best_only=train_config.save_best_only,
    )

    # Load best model checkpoint for MLflow logging
    model_path = Path(train_config.save_dir) / "best_model.pth"
    print(f"Loading best checkpoint from {model_path} for MLflow logging")
    model.load(model_path)

    # Create example input for MLflow signature
    example_input = torch.randn(
        1,
        3,
        data_config.image_size,
        data_config.image_size,
        device="cpu",
    )
    model.eval()
    with torch.no_grad():
        example_output = model(example_input.to(model.device)).cpu().numpy()

    example_input_numpy = example_input.cpu().numpy()
    signature = infer_signature(
        example_input_numpy,
        example_output,
    )

    # Log model to MLflow with registered model name
    artifact_path = train_config.mlflow_model_name
    print(
        f"Logging trained model to MLflow at artifact path '{artifact_path}'",
    )
    mlflow.pytorch.log_model(
        pytorch_model=model,
        artifact_path=artifact_path,
        input_example=example_input_numpy,
        signature=signature,
        registered_model_name=train_config.mlflow_model_name,
    )

    return model

In [5]:
from typing import Tuple

import torch.nn as nn
from torch.utils.data import DataLoader
from zenml.materializers import BuiltInMaterializer

from src.config import EvaluationConfig
from src.steps.evaluate_model import evaluate_model


@step
def model_evaluation(
    model: nn.Module,
    test_loader: DataLoader,
) -> Tuple[float, float, float, float]:
    accuracy, precision, recall, f1 = evaluate_model(model, test_loader)
    return accuracy, precision, recall, f1


In [6]:
@step(output_materializers=BuiltInMaterializer)
def deployment_trigger(
    accuracy: float,
    precision: float,
    recall: float,
    f1: float,
) -> bool:
    with open("src/steps/test_config.yaml", "r") as f:
        config = yaml.safe_load(f)
        evaluation_config = EvaluationConfig(**config["evaluation"])

    precision_threshold_met = precision >= evaluation_config.min_precision
    recall_threshold_met = recall >= evaluation_config.min_recall
    f1_threshold_met = f1 >= evaluation_config.min_f1
    accuracy_threshold_met = accuracy >= evaluation_config.min_accuracy

    print(
        f"Deployment gate results - precision: {precision:.4f} (>= {evaluation_config.min_precision:.4f}: {precision_threshold_met}), "
        f"recall: {recall:.4f} (>= {evaluation_config.min_recall:.4f}: {recall_threshold_met}), "
        f"f1: {f1:.4f} (>= {evaluation_config.min_f1:.4f}: {f1_threshold_met}), "
        f"accuracy: {accuracy:.4f} (>= {evaluation_config.min_accuracy:.4f}: {accuracy_threshold_met})"
    )

    conditions_met = (
        precision_threshold_met
        and recall_threshold_met
        and f1_threshold_met
        and accuracy_threshold_met
    )

    return conditions_met

In [7]:
# Run the pipeline
classification_pipeline()


Initiating a new run for the pipeline: classification_pipeline.
Caching is disabled by default for classification_pipeline.
Using user: default
Using stack: local_mlflow_stack
  orchestrator: default
  model_deployer: mlflow
  experiment_tracker: mlflow_tracker
  deployer: default
  artifact_store: default
Dashboard URL for Pipeline Run: http://127.0.0.1:8237/projects/default/runs/fc2c5316-85d5-470c-a0f8-c4a9a0ab1352
Step load_data has started.
[load_data] Shape of the dataset: (9999, 5)
[load_data] Number of unique common_name: 356
[load_data] Number of missing values in image_url: 0
[load_data] Shape of the dataset after removing entries without image URLs: (9999, 5)
[load_data] Shape of the dataset before removing duplicate image URLs: (9999, 5)
[load_data] Shape of the dataset after removing duplicate image URLs: (9999, 5)
[load_data] Shape of the dataset containing common_name with less than 10 counts: (668, 5)
[load_data] Number of unique common_name with less than 10 counts: 176

2025/11/02 13:25:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[model_training] Logging trained model to MLflow at artifact path 'animal-classifier-resnet18'


2025/11/02 13:26:07 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'animal-classifier-resnet18' already exists. Creating a new version of this model...
[model_training] Registered model 'animal-classifier-resnet18' already exists. Creating a new version of this model...
Created version '2' of model 'animal-classifier-resnet18'.
[model_training] Created version '2' of model 'animal-classifier-resnet18'.


[model_training] Failed to disable MLflow autologging for the following frameworks: ['fastai'].
Step model_training has finished in 1m9s.
Step model_evaluation has started.
[model_evaluation] Generating predictions on test data...
[model_evaluation] Accuracy: 0.3086.
[model_evaluation] Macro Precision: 0.0840.
[model_evaluation] Macro Recall: 0.0785.
[model_evaluation] Macro F1: 0.0676.
[model_evaluation] Evaluation completed successfully.
Step model_evaluation has finished in 3.672s.
Step deployment_trigger has started.
[deployment_trigger] Deployment gate results - precision: 0.0840 (>= 0.0500: True), recall: 0.0785 (>= 0.0500: True), f1: 0.0676 (>= 0.0500: True), accuracy: 0.3086 (>= 0.0500: True)
Step deployment_trigger has finished in 0.392s.
Step mlflow_model_deployer_step has started.
[mlflow_model_deployer_step] Existing model server found for animal-classifier-resnet18 with the exact same configuration. Returning the existing service named zenml-animal-classifier-resnet50.
[ml

Step mlflow_model_deployer_step has finished in 0.913s.
Pipeline run has finished in 1m23s.


PipelineRunResponse(body=PipelineRunResponseBody(created=datetime.datetime(2025, 11, 2, 18, 24, 55, 994353), updated=datetime.datetime(2025, 11, 2, 18, 26, 19, 419273), user_id=UUID('1d58d1cd-32ec-42c4-87e1-d00884e8fd9f'), project_id=UUID('6a3191a0-28e1-4f65-919d-8cb27a096ded'), status=<ExecutionStatus.COMPLETED: 'completed'>, in_progress=False, status_reason=None), metadata=PipelineRunResponseMetadata(run_metadata={}, config=PipelineConfiguration(enable_cache=False, enable_artifact_metadata=None, enable_artifact_visualization=None, enable_step_logs=None, environment={}, secrets=[], enable_pipeline_logs=None, execution_mode=<ExecutionMode.CONTINUE_ON_FAILURE: 'continue_on_failure'>, settings={}, tags=None, extra={}, failure_hook_source=None, success_hook_source=None, init_hook_source=None, init_hook_kwargs=None, cleanup_hook_source=None, model=None, parameters=None, retry=None, substitutions={'date': '2025_11_02', 'time': '18_24_55_960340'}, cache_policy=None, name='classification_pipe

## Inspecting Deployment Services

The message you saw means the deployer found an existing service with the same configuration and is reusing it (efficient!). However, the service name mentions "resnet50" which might be from a previous deployment.


In [ ]:
# Test the prediction service with a sample input
import numpy as np

if services and services[0].is_running:
    service = services[0]
    print(f"Testing prediction service at: {service.prediction_url}\n")

    # Create a dummy input (batch_size=1, channels=3, height=224, width=224)
    dummy_input = np.random.randn(1, 3, 224, 224).astype(np.float32)

    try:
        prediction = service.predict(dummy_input)
        print(f"✓ Service is responding!")
        print(f"  Input shape: {dummy_input.shape}")
        print(f"  Output shape: {prediction.shape}")
        print(f"  Predicted class: {np.argmax(prediction)}")
    except Exception as e:
        print(f"✗ Error during prediction: {e}")
else:
    print("No running services found. The service might be starting up...")


In [ ]:
# Optional: Clean up old/stopped services
# Uncomment and run if you want to remove old services

# for service in services:
#     if not service.is_running:
#         print(f"Stopping and removing service: {service.config.service_name}")
#         service.stop(timeout=10)
#         model_deployer.delete_service(service.uuid)
#     else:
#         print(f"Service {service.config.service_name} is running, keeping it.")

print("To clean up services, uncomment the code above and run this cell.")


### What's Happening?

**The message you saw is NORMAL behavior:**

1. **Service Reuse**: The MLflow deployer found an existing service with the same configuration (pipeline name, step name, model name) and is reusing it to save resources.

2. **Name Mismatch**: The service name shows "resnet50" but you're deploying "resnet18" - this is just the service's historical name from when it was first created. The actual model being served should be updated.

3. **Why This Happens**: 
   - When you run the pipeline multiple times, ZenML checks if a compatible service already exists
   - If found, it updates the model in that service rather than creating a new one
   - This is more efficient and avoids port conflicts

**What You Should Do:**

✅ **If the deployment decision was TRUE**: The service should be running and serving your new model
- Run Cell 8 to check service status
- Run Cell 9 to test predictions

❌ **If the deployment decision was FALSE**: The existing service was kept but NOT updated with the new model
- Check the deployment trigger output to see why
- The old model is still being served

**To Force a Fresh Deployment:**
- Stop and delete old services using Cell 10
- Or change the `model_name` parameter in the pipeline to create a separate service


In [9]:
# Check all deployed services
from zenml.integrations.mlflow.model_deployers.mlflow_model_deployer import (
    MLFlowModelDeployer,
)

# model_deployer = MLFlowModelDeployer.get_active_model_deployer()
mlflow.set_active_model(model_id="m-2325c9bb170f4e0ab05a76109c861f3e")


# # Find all services for this pipeline
# services = model_deployer.find_model_server(
#     pipeline_name="classification_pipeline",
#     pipeline_step_name="mlflow_model_deployer_step",
#     running=False,  # Include both running and stopped services
# )

# print(f"Found {len(services)} service(s):\n")
# for i, service in enumerate(services):
#     print(f"Service {i + 1}:")
#     print(f"  Name: {service.config.service_name}")
#     print(f"  Model Name: {service.config.model_name}")
#     print(f"  Model URI: {service.config.model_uri}")
#     print(f"  Running: {service.is_running}")
#     print(f"  Status: {service.status.state}")
#     if service.is_running:
#         print(f"  Prediction URL: {service.prediction_url}")
#     print()


2025/11/02 13:30:04 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-2325c9bb170f4e0ab05a76109c861f3e


LoggedModel()

In [8]:
train_dataloader, validation_dataloader, test_dataloader = load_data()

model = AnimalClassifierResNet18(
    num_classes=5,
    optimizer="adamw",
    pretrained=True,
    lr=0.001,
    max_lr=0.01,
    epochs=10,
    device="cuda",
    train_loader=train_dataloader,
    class_weights=None,
)

Running single step pipeline to execute step load_data
The unlisted option is deprecated and will be removed in a future version. Every run will always be associated with a pipeline.
Initiating a new run for the pipeline: load_data.
Registered new pipeline: load_data.
Caching is disabled by default for load_data.
Using user: default
Using stack: local_mlflow_stack
  orchestrator: default
  model_deployer: mlflow
  experiment_tracker: mlflow_tracker
  deployer: default
  artifact_store: default
Dashboard URL for Pipeline Run: http://127.0.0.1:8237/projects/default/runs/866d49d3-aac0-49ec-ae8f-35a963c7db72
Step load_data has started.
[load_data] Shape of the dataset: (9999, 5)
[load_data] Number of unique common_name: 356
[load_data] Number of missing values in image_url: 0
[load_data] Shape of the dataset after removing entries without image URLs: (9999, 5)
[load_data] Shape of the dataset before removing duplicate image URLs: (9999, 5)
[load_data] Shape of the dataset after removing du

In [9]:
type(model.)

╭──────────────────────────────────────────────────────────────────────────────────────────────────╮
│ type(model.)                                                                                     │
│            ▲                                                                                     │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
SyntaxError: invalid syntax